# جلسه ۳: خروجی‌های ساختاریافته و تجزیه خروجی

## اهداف
- گرفتن خروجی JSON قابل اعتماد از LLMها
- استفاده از حالت JSON و شِماهای خروجی ساختاریافته
- تجزیه و اعتبارسنجی پاسخ‌های LLM به صورت برنامه‌نویسی
- مدیریت خروجی‌های نامعتبر به صورت ظریف

**مدت زمان:** ۴۰ دقیقه | **سطح:** متوسط

**چرا این مهم است:** اپلیکیشن‌های واقعی به داده ساختاریافته (نه متن آزاد) از LLMها نیاز دارند — برای پایگاه‌های داده، APIها، رندر UI و غیره.

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

# بارگذاری متغیرهای محیطی از فایل .env
load_dotenv(dotenv_path=os.path.join("..", ".env"))

client = OpenAI()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

print(f"راه‌اندازی کامل شد! مدل: {MODEL}")

Setup complete! Model: gpt-5-mini


## ۱. مشکل: خروجی غیرساختاریافته

بدون هیچ اجبار قالبی، LLMها متن آزادی برمی‌گردانند که تجزیه برنامه‌نویسی آن دشوار است.

In [ ]:
# بدون ساختار — مدل متن آزاد برمی‌گرداند
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Extract: name, email, and company from: 'John Smith from Acme Corp (john@acme.com)'"}],
    temperature=0
)

result = response.choices[0].message.content
print("خروجی متن آزاد:")
print(result)
print(f"\nنوع: {type(result)}")
# این فقط یک رشته است — استفاده در کد دشوار است!

Free text output:
```json
{
  "name": "John Smith",
  "email": "john@acme.com",
  "company": "Acme Corp"
}
```

Type: <class 'str'>


## ۲. حالت JSON

`response_format={"type": "json_object"}` را تنظیم کنید تا مدل مجبور به خروجی JSON معتبر شود.

**مهم:** هنگام استفاده از حالت JSON، باید «JSON» را در پرامپت خود نیز ذکر کنید.

In [ ]:
# حالت JSON — خروجی JSON معتبر را تضمین می‌کند
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Extract contact information. Respond in JSON format."},
        {"role": "user", "content": "John Smith from Acme Corp (john@acme.com)"}
    ],
    response_format={"type": "json_object"},
    temperature=0
)

result = response.choices[0].message.content
print("رشته JSON:")
print(result)

# حالا می‌توانیم آن را به دیکشنری پایتون تجزیه کنیم!
data = json.loads(result)
print(f"\nداده تجزیه‌شده: {data}")
print(f"نام: {data.get('name')}")
print(f"ایمیل: {data.get('email')}")

JSON string:
```json
{
  "name": "John Smith",
  "organization": "Acme Corp",
  "email": "john@acme.com",
  "phone": null
}
```


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

## ۳. خروجی‌های ساختاریافته با شِمای JSON

حالت JSON خروجی JSON معتبر را تضمین می‌کند، اما **ساختار** ممکن است متفاوت باشد.
با `json_schema`، شما **شِمای دقیقی** تعریف می‌کنید که مدل باید از آن پیروی کند.

این به شما می‌دهد:
- نام فیلدهای تضمین‌شده
- انواع داده صحیح
- فیلدهای اجباری در مقابل اختیاری

In [ ]:
# تعریف شِمای سخت‌گیرانه برای بررسی فیلم
movie_review_schema = {
    "type": "json_schema",
    "json_schema": {
        "name": "movie_review_analysis",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "movie_title": {"type": "string"},
                "sentiment": {"type": "string", "enum": ["positive", "negative", "mixed"]},
                "rating": {"type": "number"},
                "key_themes": {"type": "array", "items": {"type": "string"}},
                "summary": {"type": "string"}
            },
            "required": ["movie_title", "sentiment", "rating", "key_themes", "summary"],
            "additionalProperties": False
        }
    }
}

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Analyze movie reviews and extract structured data."},
        {"role": "user", "content": """The new Dune movie was visually stunning with incredible cinematography.
        The acting was superb, especially Timothee Chalamet. However, the pacing
        was slow in the middle. Overall a great sci-fi epic. 8/10."""}
    ],
    response_format=movie_review_schema,
    temperature=0
)

data = json.loads(response.choices[0].message.content)
print(json.dumps(data, indent=2))

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
# دسترسی برنامه‌نویسی به داده ساختاریافته
print(f"فیلم: {data['movie_title']}")
print(f"احساسات: {data['sentiment']}")
print(f"امتیاز: {data['rating']}")
print(f"مضامین: {', '.join(data['key_themes'])}")
print(f"خلاصه: {data['summary']}")

NameError: name 'data' is not defined

## ۴. مثال عملی: خط لوله استخراج داده

بیایید تابعی قابل استفاده مجدد بسازیم که داده ساختاریافته را از هر متنی با استفاده از شِما استخراج کند.

In [ ]:
def extract_structured_data(text, schema, instruction="Extract the requested information."):
    """استخراج داده ساختاریافته از متن با استفاده از شِمای JSON."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": instruction},
            {"role": "user", "content": text}
        ],
        response_format=schema,
        temperature=0
    )
    return json.loads(response.choices[0].message.content)

# شِما برای استخراج اطلاعات رویداد
event_schema = {
    "type": "json_schema",
    "json_schema": {
        "name": "event_info",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "event_name": {"type": "string"},
                "date": {"type": "string"},
                "location": {"type": "string"},
                "organizer": {"type": "string"},
                "is_free": {"type": "boolean"}
            },
            "required": ["event_name", "date", "location", "organizer", "is_free"],
            "additionalProperties": False
        }
    }
}

# استخراج از متن زبان طبیعی
text = """Join us for the Annual AI Conference on March 15, 2025 at the
Convention Center in San Francisco. Organized by TechForward Inc.
Tickets start at $299."""

event = extract_structured_data(text, event_schema)
print(json.dumps(event, indent=2))

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

## ۵. پردازش چندین آیتم

یک الگوی رایج: پردازش دسته‌ای آیتم‌ها و جمع‌آوری نتایج ساختاریافته.

In [ ]:
# پردازش دسته‌ای با خروجی ساختاریافته
emails = [
    "Hi, I'd like to cancel my subscription effective immediately. - Mike",
    "When will the new features be released? I'm excited! - Sarah",
    "Your product crashed and I lost all my data. This is unacceptable! - Tom"
]

email_schema = {
    "type": "json_schema",
    "json_schema": {
        "name": "email_classification",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "sender": {"type": "string"},
                "intent": {"type": "string", "enum": ["cancellation", "inquiry", "complaint", "feedback"]},
                "urgency": {"type": "string", "enum": ["low", "medium", "high"]},
                "summary": {"type": "string"}
            },
            "required": ["sender", "intent", "urgency", "summary"],
            "additionalProperties": False
        }
    }
}

results = []
for email in emails:
    result = extract_structured_data(email, email_schema, "Classify this customer email.")
    results.append(result)
    print(f"  {result['sender']}: {result['intent']} (فوریت: {result['urgency']})")

print(f"\n{len(results)} ایمیل پردازش شد")

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

## ۶. مدیریت خطا

هنگام کار با خروجی‌های LLM همیشه خطاهای احتمالی را مدیریت کنید.

In [ ]:
def safe_extract(text, schema, instruction="Extract information."):
    """استخراج داده ساختاریافته با مدیریت خطا."""
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": instruction},
                {"role": "user", "content": text}
            ],
            response_format=schema,
            temperature=0
        )
        
        # بررسی اینکه آیا مدل رد شده (فیلتر محتوا)
        if response.choices[0].finish_reason == "content_filter":
            return {"error": "محتوا فیلتر شد"}
        
        return json.loads(response.choices[0].message.content)
    
    except json.JSONDecodeError as e:
        return {"error": f"تجزیه JSON ناموفق: {e}"}
    except Exception as e:
        return {"error": f"خطای API: {e}"}

# تست با ورودی عادی
result = safe_extract(
    "Meeting with Dr. Lisa Park on Friday at 2pm in Room 301",
    event_schema,
    "Extract event details."
)
print(json.dumps(result, indent=2))

{
  "error": "Failed to parse JSON: Expecting value: line 1 column 1 (char 0)"
}


## تمرین: استخراج‌کننده اطلاعات رزومه

خط لوله‌ای بسازید که اطلاعات ساختاریافته را از متن رزومه/CV استخراج کند.

In [ ]:
# تعریف شِمای رزومه
resume_schema = {
    "type": "json_schema",
    "json_schema": {
        "name": "resume_data",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "name": {"type": "string"},
                "email": {"type": "string"},
                "years_of_experience": {"type": "number"},
                "skills": {"type": "array", "items": {"type": "string"}},
                "education": {"type": "string"},
                "current_role": {"type": "string"}
            },
            "required": ["name", "email", "years_of_experience", "skills", "education", "current_role"],
            "additionalProperties": False
        }
    }
}

resume_text = """Jane Doe | jane.doe@email.com
Senior Machine Learning Engineer at Google (5 years experience)
Previously: Data Scientist at Meta (2 years)
Education: M.S. Computer Science, Stanford University
Skills: Python, PyTorch, TensorFlow, NLP, Computer Vision, MLOps, SQL"""

result = safe_extract(resume_text, resume_schema, "Extract resume information.")
print(json.dumps(result, indent=2))

# حالا می‌توانیم از این داده به صورت برنامه‌نویسی استفاده کنیم
print(f"\nکاندیدا: {result['name']}")
print(f"تجربه: {result['years_of_experience']} سال")
print(f"مهارت‌های برتر: {', '.join(result['skills'][:3])}")

{
  "error": "Failed to parse JSON: Expecting value: line 1 column 1 (char 0)"
}


KeyError: 'name'

## خلاصه

| روش | کاربرد |
|--------|----------|
| **حالت JSON** | خروجی ساختاریافته ساده، شِمای انعطاف‌پذیر |
| **شِمای JSON** | ساختار سخت‌گیرانه با فیلدها و انواع تعریف‌شده |
| **مدیریت خطا** | اپلیکیشن‌های تولیدی که به قابلیت اطمینان نیاز دارند |

**نکات کلیدی:**
- هنگام نیاز به داده ساختاریافته همیشه از `response_format` استفاده کنید
- `json_schema` با `strict: True` قابل اطمینان‌ترین نتایج را می‌دهد
- فراخوانی‌های API را در مدیریت خطا برای استفاده تولیدی قرار دهید

**جلسه بعدی:** امبدینگ‌های متنی و جستجوی معنایی!